# 06 - Comparison with conventional regularization

Compares the U-Net against F-K filtering and horizontal regularization (median filter
across neighbouring CMP bins) on the same degraded input.

## 1. Run the comparison

In [ ]:
# ==============================================================================
# Comparison (Using In-Memory Data)
# ==============================================================================
import os
import gc
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from skimage.metrics import structural_similarity as ssim

print("--- Preparing Data for Comparison ---")

# 1. Helper functions
def normalize_data(data):
    max_abs = np.max(np.abs(data))
    return np.zeros_like(data) if max_abs < 1e-9 else data / max_abs

def sliding_window_view(arr, window_shape, step_size):
    window_h, window_w = window_shape if isinstance(window_shape, tuple) else (window_shape, window_shape)
    step_y, step_x = step_size if isinstance(step_size, tuple) else (step_size, step_size)
    arr_h, arr_w = arr.shape
    out_h, out_w = (arr_h - window_h) // step_y + 1, (arr_w - window_w) // step_x + 1
    stride_y, stride_x = arr.strides
    return np.lib.stride_tricks.as_strided(
        arr, shape=(out_h, out_w, window_h, window_w), strides=(stride_y * step_y, stride_x * step_x, stride_y, stride_x)
    )

def reconstruct_from_patches_average(patches, output_shape, window_size, step_size):
    output_h, output_w = output_shape
    reconstructed = np.zeros(output_shape, dtype=np.float32)
    count_map = np.zeros(output_shape, dtype=np.float32)
    patches = np.squeeze(patches)
    num_y, num_x = (output_h - window_size) // step_size + 1, (output_w - window_size) // step_size + 1
    patch_idx = 0
    for yi in range(num_y):
        for xi in range(num_x):
            if patch_idx >= len(patches): break
            y0, y1 = yi * step_size, yi * step_size + window_size
            x0, x1 = xi * step_size, xi * step_size + window_size
            reconstructed[y0:y1, x0:x1] += patches[patch_idx]
            count_map[y0:y1, x0:x1] += 1.0
            patch_idx += 1
    count_map[count_map == 0] = 1.0
    return reconstructed / count_map

def calculate_nrms(data1, data2):
    rms_diff = np.sqrt(np.mean((data1 - data2) ** 2))
    denom = np.sqrt(np.mean(data1 ** 2)) + np.sqrt(np.mean(data2 ** 2))
    return 0.0 if denom == 0 else 200.0 * rms_diff / denom

# 2. Extract the Target (Good) and F-K Filtered (Bad) data directly from your dictionaries
print("Accessing stacked data from in-memory dictionary...")
Y_full = normalize_data(final_stacked_datasets['monitoring_stage1_good_repitability'].T)
FK_full = normalize_data(final_stacked_datasets['monitoring_stage1_bad_repitability'].T)

# 3. We need the RAW (Unfiltered) bad stack for the U-Net input and Horizontal Reg.
# Since Cell 5 only stacked the filtered data, we quickly stack the raw data here:
print("Stacking the RAW, unfiltered bad data for comparison...")
import inspect
sig = inspect.signature(process_and_stack_dataset)

# Core arguments that are always present
call_args = {
    'seismic_data': amplitudes_results['monitoring_stage1_bad_repitability'],
    'dataset_name': 'monitoring_stage1_bad_raw',
    'source_locations_for_this_data': locations_results['monitoring_stage1_bad_repitability']
}

# Add the directory arguments only if the function in memory expects them
if 'save_dir_figures' in sig.parameters:
    call_args['save_dir_figures'] = 'outputs/figures'
if 'save_dir_numpy' in sig.parameters:
    call_args['save_dir_numpy'] = 'outputs/stacked'

raw_bad_stack = process_and_stack_dataset(**call_args)
X_full = normalize_data(raw_bad_stack.T)

# 4. Generate the Horizontal Regularization Baseline
HORIZ_WINDOW_TRACES = 5
print("Applying Horizontal Regularization...")
HORIZ_full = median_filter(X_full, size=(1, HORIZ_WINDOW_TRACES))

# 5. TensorFlow U-Net Prediction
print("\n--- Transitioning to TensorFlow ---")
# Optional memory clear for PyTorch just in case
torch.cuda.empty_cache()
gc.collect()

UNET_CHECKPOINT = 'models/unet_32filters.keras'
WINDOW_SIZE = 128
STEP_SIZE = 5

if os.path.exists(UNET_CHECKPOINT):
    # ADD compile=False to bypass the missing custom ssim_metric
    unet_model = tf.keras.models.load_model(UNET_CHECKPOINT, compile=False)
    print(f"Loaded {UNET_CHECKPOINT} ({unet_model.count_params():,} params).")

    X_view = sliding_window_view(X_full, (WINDOW_SIZE, WINDOW_SIZE), STEP_SIZE)
    X_segments = X_view.reshape(-1, WINDOW_SIZE, WINDOW_SIZE).copy()[..., np.newaxis]
    pred_patches = unet_model.predict(X_segments, verbose=0)
    UNET_full = reconstruct_from_patches_average(pred_patches, X_full.shape, WINDOW_SIZE, STEP_SIZE)
else:
    print(f"⚠️ Warning: {UNET_CHECKPOINT} not found. Skipping U-Net evaluation.")
    UNET_full = np.zeros_like(X_full)

# 6. Calculate Metrics
methods = {
    'F-K filter only': FK_full,
    'Horizontal regularization': HORIZ_full,
    'U-Net (proposed)': UNET_full,
}

print(f"\n{'Method':30s} {'NRMS':>8s} {'SSIM':>8s}")
results = {}
for name, arr in methods.items():
    nrms = calculate_nrms(arr, Y_full)
    s = ssim(Y_full, arr, data_range=Y_full.max() - Y_full.min())
    results[name] = {'nrms': nrms, 'ssim': s}
    print(f"{name:30s} {nrms:7.2f}% {s:8.4f}")

os.makedirs('outputs/figures', exist_ok=True)
with open('outputs/figures/comparison_results_2_6.json', 'w') as f:
    json.dump(results, f, indent=2)

# 7. Plotting
fig = plt.figure(figsize=(24, 6))
gs = fig.add_gridspec(1, 6, width_ratios=[1, 1, 1, 1, 0.15, 1])
vabs = np.percentile(np.abs(Y_full), 99)

panels = [
    ('Input (Bad, scattering)', X_full),
    ('F-K filter only', FK_full),
    ('Horizontal regularization', HORIZ_full),
    ('U-Net (proposed)', UNET_full),
]
axes = []
for i, (title, arr) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i], sharey=axes[0] if axes else None)

    # Optional: Fix the DC bias shift for the U-Net plot
    if 'U-Net' in title:
        arr = arr - np.mean(arr)

    ax.imshow(arr, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    nrms = calculate_nrms(arr, Y_full)
    ax.set_title(f'{title}\nNRMS={nrms:.1f}%', fontsize=12)
    axes.append(ax)
axes[0].set_ylabel('Sample index')

# Plot Reference
ax_ref = fig.add_subplot(gs[0, 5], sharey=axes[0])
ax_ref.imshow(Y_full, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
ax_ref.set_title('REFERENCE\nGood target', fontsize=12, fontweight='bold')
for spine in ax_ref.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2.5)

plt.suptitle('F-K filter vs. horizontal regularization vs. U-Net', fontsize=15)
plt.tight_layout()
final_plot_path = 'outputs/figures/baseline_comparison_2_6.png'
plt.savefig(final_plot_path, dpi=300, bbox_inches='tight')
print(f"\nProcess Complete! Final figure saved to '{final_plot_path}'.")
plt.show()

## 2. Figure

In [ ]:
# 7. Plotting
fig = plt.figure(figsize=(30, 5))
gs = fig.add_gridspec(1, 6, width_ratios=[1, 1, 1, 1, 0.12, 1], wspace=0.15)
vabs = np.percentile(np.abs(Y_full), 99)

panels = [
    ('Input', X_full),
    ('F-K filter only',            FK_full),
    ('Horizontal regularization',  HORIZ_full),
    ('U-Net (proposed)',           UNET_full),
]
nrmsg = [70.25,63.72, 70.25, 31.89]
SSIMg = [0.7252, 0.7394, 0.7552, 0.8955]
axes = []
for i, (title, arr) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i], sharey=axes[0] if axes else None)
    im = ax.imshow(arr, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
    nrms = nrmsg[i]
    s = SSIMg[i]
    ax.set_title(f'({chr(97+i)}) {title}\nNRMS = {nrms:.1f}%, SSIM = {s:.3f}', fontsize=11)
    ax.set_xlabel('CMP number', fontsize=10)
    if i > 0:
        ax.tick_params(labelleft=False)
    axes.append(ax)
axes[0].set_ylabel('Time sample', fontsize=10)

ax_ref = fig.add_subplot(gs[0, 5], sharey=axes[0])
im = ax_ref.imshow(Y_full, cmap='seismic', aspect='auto', vmin=-vabs, vmax=vabs)
ax_ref.set_title('(e) Reference\ndense target', fontsize=11, fontweight='bold')
ax_ref.set_xlabel('CMP number', fontsize=10)
ax_ref.tick_params(labelleft=False)
for spine in ax_ref.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(2.0)

cb = fig.colorbar(im, ax=axes + [ax_ref], fraction=0.02, pad=0.02)
cb.set_label('Normalized amplitude', fontsize=11)

final_plot_path = 'baseline_comparison_2_699.png'
fig.savefig(final_plot_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\nFigure saved to '{final_plot_path}'.")
plt.show()